In [0]:
bronze_path   = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/bronze/'
silver_path   = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/silver/'
gold_path     = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/gold/'
resource_path = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/resource/'

In [0]:

# criar várias tabelas temporárias de forma prática
silver_map = {
   
    "tmp_silver_customers":  f"{silver_path}/customers/",
    "tmp_silver_orders":     f"{silver_path}/orders/",
    "tmp_silver_products":   f"{silver_path}/product/"

}

for view_name, path in silver_map.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )


In [0]:
df_sales_ny_gold = spark.sql("""

select
    shipped_date,
    round(sum(total_sale),2) as total_sale
from tmp_silver_orders
where state = 'NY'
and status = 'Delivered'
and shipped_date is not null
group by shipped_date

""" )


#salvar em parquet como delta tabela de teste SILVER
df_sales_ny_gold.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .save(f'{gold_path}/sales_ny')